# 07 — Pyramide des âges INSEE

Analyse de la pyramide des âges 1991–2070 (données observées + projections) croisée avec les effectifs CNAV.

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sqlalchemy as sa
from src.db import engine

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10

conn = engine().connect()

## 1. Pyramide tornado — années clés

In [ ]:
pyr = pd.read_sql("""
    SELECT annee, age, genre_code, population
    FROM ext.INSEE_PyramideAges
    WHERE genre_code IN ('H','F') AND annee IN (1991, 2006, 2023, 2040)
    ORDER BY annee, genre_code, age
""", conn)

annees = sorted(pyr['annee'].unique())
fig, axes = plt.subplots(1, len(annees), figsize=(18, 7), sharey=True)

for ax, annee in zip(axes, annees):
    df = pyr[pyr['annee'] == annee]
    h  = df[df['genre_code'] == 'H'].set_index('age')['population'] / 1e6
    f  = df[df['genre_code'] == 'F'].set_index('age')['population'] / 1e6
    ages = df['age'].unique()
    ax.barh(ages, -h.reindex(ages, fill_value=0), color='steelblue', label='Hommes')
    ax.barh(ages, f.reindex(ages, fill_value=0), color='salmon', label='Femmes')
    ax.set_title(str(annee), fontsize=12, fontweight='bold')
    ax.axhline(60, color='red', lw=0.8, ls='--', alpha=0.6)
    ax.set_xlabel('Millions')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{abs(x):.1f}'))
    if annee == annees[0]:
        ax.set_ylabel('Âge')
    if annee == annees[-1]:
        ax.legend()

fig.suptitle('Pyramides des âges — France (INSEE)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('reports/07_pyramide_tornado.png', bbox_inches='tight')
plt.show()

## 2. Vieillissement de la population

In [ ]:
vieill = pd.read_sql("""
    SELECT annee,
           SUM(CASE WHEN age >= 60 AND genre_code = 'H' THEN population END) AS pop60_h,
           SUM(CASE WHEN age >= 60 AND genre_code = 'F' THEN population END) AS pop60_f,
           SUM(CASE WHEN age >= 65 AND genre_code IN ('H','F') THEN population END) AS pop65_plus,
           SUM(CASE WHEN age >= 75 AND genre_code IN ('H','F') THEN population END) AS pop75_plus,
           SUM(CASE WHEN age BETWEEN 20 AND 64 AND genre_code IN ('H','F') THEN population END) AS pop_actif,
           SUM(CASE WHEN genre_code IN ('H','F') THEN population END) AS pop_totale
    FROM ext.INSEE_PyramideAges
    GROUP BY annee
    HAVING SUM(CASE WHEN genre_code IN ('H','F') THEN population END) > 0
    ORDER BY annee
""", conn)

vieill['ratio_dependance'] = (vieill['pop65_plus'] / vieill['pop_actif'] * 100).round(1)
vieill['part_60plus']      = ((vieill['pop60_h'] + vieill['pop60_f']) / vieill['pop_totale'] * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.fill_between(vieill['annee'], vieill['part_60plus'], alpha=0.4, color='darkorange')
ax1.plot(vieill['annee'], vieill['part_60plus'], color='darkorange')
ax1.set_title('Part des 60 ans et plus (%)')
ax1.set_ylabel('%'); ax1.grid(alpha=0.3)
ax1.axvline(2024, ls='--', color='gray', lw=0.8, label='Proj →')
ax1.legend()

ax2.plot(vieill['annee'], vieill['ratio_dependance'], color='crimson')
ax2.set_title('Ratio de dépendance démographique (65+/20-64, %)')
ax2.set_ylabel('%'); ax2.grid(alpha=0.3)
ax2.axvline(2024, ls='--', color='gray', lw=0.8)

plt.tight_layout()
plt.savefig('reports/07_vieillissement.png', bbox_inches='tight')
plt.show()

print(vieill[['annee','part_60plus','ratio_dependance']].tail(10).to_string(index=False))

## 3. Baby-boom 1946–1964 et vague de départs à la retraite

In [ ]:
# Population née entre 1946 et 1964 : suivi par décennie
boom = pd.read_sql("""
    SELECT annee,
           SUM(CASE WHEN genre_code IN ('H','F') THEN population END) AS pop_boom
    FROM ext.INSEE_PyramideAges
    WHERE age BETWEEN (annee - 1964) AND (annee - 1946)
      AND annee BETWEEN 1991 AND 2040
      AND genre_code IN ('H','F')
    GROUP BY annee
    ORDER BY annee
""", conn)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(boom['annee'], boom['pop_boom'] / 1e6, color='steelblue', alpha=0.8)
ax.set_title('Effectifs de la génération baby-boom (nés 1946-1964) par année', fontweight='bold')
ax.set_ylabel('Millions de personnes')
ax.set_xlabel('Année')
ax.axvline(2006, color='red', ls='--', lw=0.9, label='2006 : premiers 60 ans')
ax.axvline(2024, color='orange', ls='--', lw=0.9, label='2024 : derniers 60 ans (64 ans réforme Borne)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('reports/07_babyboom.png', bbox_inches='tight')
plt.show()

## 4. Croisement CNAV × INSEE — couverture retraite

In [ ]:
couv = pd.read_sql("""
    SELECT annee, genre, retraites_cnav, population_60_plus, taux_retraite_pct
    FROM rpt.v_TauxRetraiteVsPopulation
    WHERE annee BETWEEN 1991 AND 2023 AND population_60_plus IS NOT NULL
    ORDER BY annee, genre
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for gc, color in [('Femmes', 'salmon'), ('Hommes', 'steelblue')]:
    sub = couv[couv['genre'] == gc]
    axes[0].plot(sub['annee'], sub['retraites_cnav'] / 1e6, label=gc, color=color)
    axes[1].plot(sub['annee'], pd.to_numeric(sub['taux_retraite_pct'], errors='coerce'), label=gc, color=color)

axes[0].set_title('Effectifs CNAV (M)'); axes[0].set_ylabel('Millions'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Taux de retraite CNAV / pop. 60+ (%)'); axes[1].set_ylabel('%'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/07_couverture_retraite.png', bbox_inches='tight')
plt.show()

## 5. Projection pression démographique 2024–2050

In [ ]:
proj = pd.read_sql("""
    SELECT annee,
           SUM(CASE WHEN age >= 65 AND genre_code IN ('H','F') THEN population END) * 1.0
           / NULLIF(SUM(CASE WHEN age BETWEEN 20 AND 64 AND genre_code IN ('H','F') THEN population END), 0)
               AS ratio_dep,
           SUM(CASE WHEN age >= 60 AND genre_code IN ('H','F') THEN population END) AS pop60plus
    FROM ext.INSEE_PyramideAges
    WHERE annee BETWEEN 1991 AND 2050 AND genre_code IN ('H','F')
    GROUP BY annee
    ORDER BY annee
""", conn)

# Jalons des réformes
reformes = {1993: 'Balladur', 2003: 'Fillon', 2010: 'Woerth', 2014: 'Touraine', 2023: 'Borne'}

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(proj['annee'], proj['ratio_dep'] * 100, color='crimson', lw=2)
ax.fill_between(proj['annee'], proj['ratio_dep'] * 100, alpha=0.15, color='crimson')
ax.axvline(2024, color='gray', ls=':', lw=1, label='Début projections')
for annee, nom in reformes.items():
    ax.axvline(annee, color='navy', ls='--', lw=0.7, alpha=0.7)
    ax.text(annee + 0.3, ax.get_ylim()[1] * 0.95, nom, rotation=90, fontsize=7, color='navy', va='top')
ax.set_title('Ratio de dépendance démographique (65+ / 20-64, %)', fontweight='bold')
ax.set_ylabel('%'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('reports/07_pression_demo.png', bbox_inches='tight')
plt.show()

print("\nProjections clés :")
print(proj[proj['annee'].isin([2023, 2030, 2040, 2050])][['annee','ratio_dep','pop60plus']].to_string(index=False))

In [ ]:
conn.close()